In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df_users_raw = spark.read.table('retail_dev_schema.bronze_users')

In [0]:
df_users_drop_nulls = df_users_raw.filter(F.col("id").isNotNull() | F.col("email").isNotNull() | (F.col("first_name").isNotNull() & F.col("last_name").isNotNull()))
                                          
df_users_silver = df_users_drop_nulls.withColumn("country", F.trim('country')) \
                              .withColumn("email", F.lower('email')) \
                              .withColumn("full_name", F.concat(F.col('first_name'), F.lit(" "), F.col('last_name'))) \
                              .withColumn("batch_num", F.split('batch_id', '_')[1].cast('int')) \
                              .withColumnRenamed("batch_id", "source_batch_id") \
                              .withColumn("processed_at", F.current_timestamp())
drop_cols = ["batch_num", "row", "first_name", "last_name"]
user_window = Window.partitionBy(F.col('id')).orderBy(F.col('batch_num').desc())
df_users_silver = df_users_silver.withColumn("row", F.row_number().over(user_window)).filter(F.col('row') == 1).drop(*drop_cols)

In [0]:
df_users_silver.write.mode("overwrite").saveAsTable("retail_dev_schema.silver_users")